# qPCR Auto Analysis & Plot

本 Notebook 用于交互式处理 qPCR 数据。所有核心逻辑来自 `src/` 模块。

**数据来源**: Bio-Rad CFX Maestro 导出 CSV  
**分析流程**: $\Delta Cq \rightarrow 2^{-\Delta Cq} \rightarrow$ 对照组归一化 $\rightarrow$ t-test $\rightarrow$ 绘图

## 1. 导入与配置

In [ ]:
import sys
from pathlib import Path

# Allow importing from the parent directory
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt

from src.config import load_config
from src.io import read_qpcr_csv, write_result_csv
from src.preprocessing import (
    group_by_target,
    align_target_to_ref,
    parse_sample_groups,
    get_control_mask,
)
from src.analysis import (
    calculate_delta_cq,
    calculate_fold_change,
    normalize_to_control,
    build_result_dataframe,
)
from src.statistics import run_all_comparisons
from src.visualization import plot_qpcr_results

%matplotlib inline

In [ ]:
# ---- 修改以下参数匹配你的实验 ----
INPUT_FILE = "../data/admin_2025-12-10 14-24-55_782BR20773.csv"
CONFIG_FILE = "../config.yaml"

# 分组标签映射（原始样本名 → 显示名称）
GROUP_LABELS = {
    "PFOA001": "PFOA 0.01mg/L",
    "PFOA1": "PFOA 1mg/L",
}

# 加载配置
config = load_config(CONFIG_FILE)
ref_gene = config["reference_gene"]
control_group = config["control_group"]
separator = config["replicate_separator"]
skiprows = config["skiprows"]

print(f"Reference gene: {ref_gene}")
print(f"Control group: {control_group}")
print(f"Input file: {INPUT_FILE}")

## 2. 读取数据

In [ ]:
df = read_qpcr_csv(INPUT_FILE, skiprows=skiprows)

# 查看可用的 Target 基因
print("Targets found:", list(df["Target"].unique()))
print(f"Total rows: {len(df)}")
df.head(10)

## 3. 分组与预处理

按 Target 分组，按 Sample 排序。

In [ ]:
targets = group_by_target(df)

# 分离内参基因和目的基因
ref_df = targets[ref_gene]
target_genes = {k: v for k, v in targets.items() if k != ref_gene}

print("Reference:", ref_gene)
print("Target genes:", list(target_genes.keys()))

for name, tdf in targets.items():
    n_samples = tdf["Sample"].nunique()
    print(f"  {name}: {len(tdf)} rows, {n_samples} unique samples")

## 4. 逐基因分析与绘图

In [ ]:
for gene_name, gene_df in target_genes.items():
    print(f"\n{'='*50}")
    print(f"Processing: {gene_name}")
    print("="*50)

    # 对齐目标基因与内参基因（按 Sample 位置对应）
    target_cq, ref_cq, aligned_samples = align_target_to_ref(gene_df, ref_df)

    # 从样本名解析分组
    aligned_groups = parse_sample_groups(aligned_samples, separator)
    group_list = aligned_groups.map(
        lambda g: GROUP_LABELS.get(g, g)
    ).tolist()
    control_mask = get_control_mask(aligned_groups, control_group)

    # 计算
    delta_cq = calculate_delta_cq(target_cq, ref_cq)
    fold_change = calculate_fold_change(delta_cq)
    normalized = normalize_to_control(fold_change, control_mask)

    result_df = build_result_dataframe(normalized, group_list)

    # 导出 CSV
    output_path = f"../output/{gene_name}_qPCR_result.csv"
    write_result_csv(result_df, output_path)
    print(f"Saved: {output_path}")

    # 统计检验
    display_control = GROUP_LABELS.get(control_group, control_group)
    stats_results = run_all_comparisons(result_df, display_control)
    print("\nStatistics:")
    for grp, s in stats_results.items():
        print(f"  {grp} vs {display_control}: p={s['p_value']:.6f} ({s['asterisks']})")

    # 绘图
    plot_qpcr_results(
        result_df,
        target_gene=gene_name,
        stats_results=stats_results,
        control_group=display_control,
        figure_config=config.get("figure"),
    )
    plt.show()

---
## 版本信息

- **v2.0**: 模块化重构，支持任意数量目的基因、自动分组解析、CLI + Notebook 双入口。